# درس ۰۸ - الگوی طراحی چندعاملی


## راه‌اندازی


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## چرا سیستم‌های چند-عاملی؟

کارهای دنیای واقعی مانند برنامه‌ریزی سفر شامل انواع مختلفی از تخصص‌ها است — لجستیک، دانش محلی، بودجه‌بندی، و غیره. یک عامل واحد که سعی کند همه چیز را مدیریت کند به سرعت کنترل‌ناپذیر می‌شود.

سیستم‌های چند-عاملی این مسئله را از طریق **تخصص‌گرایی** حل می‌کنند: هر عامل بر یک حوزه تخصصی تمرکز می‌کند و نتایجی با کیفیت بالاتر نسبت به یک فرد عمومی ارائه می‌دهد. آن‌ها همچنین **قابلیت مقیاس‌پذیری** را بهبود می‌بخشند — می‌توانید عوامل جدیدی اضافه کنید (مثلاً متخصص پرواز، منتقد رستوران) بدون اینکه جریان کاری موجود را بازنویسی کنید. عوامل از طریق یک خط لوله ساختار یافته ترکیب می‌شوند و زمینه را از یکی به دیگری منتقل می‌کنند.


## ایجاد عوامل تخصصی


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ساخت یک گردش کار ترتیبی

`WorkflowBuilder` به شما اجازه می‌دهد که عوامل را به صورت یک گراف جهت‌دار وصل کنید. در اینجا یک خط لوله ساده دو مرحله‌ای ایجاد می‌کنیم: **TravelPlanner** برنامه سفر را تهیه می‌کند، سپس **TravelConcierge** آن را بررسی و بهبود می‌بخشد.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## اضافه کردن عوامل بیشتر به گردش کار

یکی از بزرگ‌ترین مزایای الگوی چندعاملی، سهولت توسعه آن است. در زیر یک عامل **BudgetReviewer** اضافه می‌کنیم که طرح را براساس بودجه‌ی مسافر بررسی می‌کند، مواردی که ممکن است هزینه‌ها را از حد مجاز فراتر ببرد علامت‌گذاری می‌کند، و جایگزین‌های صرفه‌جویی در هزینه را پیشنهاد می‌دهد. گردش کار اکنون سه عامل را به ترتیب اجرا می‌کند:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## خلاصه

در این درس شما یاد گرفتید چگونه:

1. **ایجاد عاملیت‌های تخصصی** — هرکدام با نقش متمرکز (برنامه‌ریزی، پذیرش، بازبینی بودجه).
2. **وصل کردن عاملیت‌ها به یک جریان کاری ترتیبی** با استفاده از `WorkflowBuilder` و `add_edge`.
3. **جریان خروجی** از یک خط لوله چندعاملی را دنبال کنید و مشخص کنید که کدام عامل در حال صحبت است.
4. **گسترش یک جریان کاری** با اضافه کردن عاملیت‌های جدید به زنجیره بدون تغییر دادن عاملیت‌های موجود.

الگوی طراحی چندعاملی هر عامل را ساده نگه می‌دارد و در عین حال نتایجی غنی‌تر و با مرور دقیق‌تر نسبت به هر عامل منفرد به تنهایی تولید می‌کند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:  
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما تلاش می‌کنیم دقت را حفظ کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است دارای خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری آن به‌عنوان مرجع معتبر باید در نظر گرفته شود. برای اطلاعات حیاتی، استفاده از ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچگونه سوتفاهم یا برداشت نادرست ناشی از استفاده این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
